# Generate client migrations

Adding an additional client for a local government requires **2 steps**, and the configuration differs between staging (TI) and production:

1. **Link the client id to the local government with a migration.** This notebook generates those migrations from the MAGDA-supplied spreadsheet.
2. **Configure the public key for each client in the beheerportaal:**
   - TI:  https://beheerportaal-ti.vlaanderen.be/
   - PRD: https://beheerportaal.vlaanderen.be/

   Navigate to *Toegangsbeheer → API- en Clientbeheer → Mijn OAuth Client → Mijn OAuth Clients*, copy the JWK config from an already-configured client (e.g. Brugge) onto the new client, and save (*Wijzigen*).

## What this notebook does (step 1)

Reads an xlsx with columns `KBOnummer, Klant, TAN, URI TEST, URI PRD, CliëntID TEST, CliëntID PRD` and writes one `.sparql` migration per (row, environment) into the appropriate `dev-migrations` directory of each deployed app.

Each migration looks up the bestuurseenheid by KBO at apply-time (no hard-coded URI) via the structured-identifier pattern:

```
?bestuur adms:identifier ?i .
?i  generiek:gestructureerdeIdentificator ?gi ;
    skos:notation "KBO nummer" .
?gi generiek:lokaleIdentificator "<KBO>" .
```

and inserts:

- `<bestuur> ext:hasSecurityScheme <oauth-config>`
- `<oauth-config> a wotsec:OAuth2SecurityScheme ; dct:identifier "<client-id>"`

The oauth-config URI is derived from the bestuurseenheid URI by swapping `/bestuurseenheden/` → `/oauth-config/`, matching the convention from the existing `brugge-client-ti` migration.

**Re-runnable:** files that already exist on disk are skipped — add new rows to the xlsx and re-run to generate only the new ones.

**Offline:** the notebook does not contact any SPARQL endpoint. KBO→bestuurseenheid resolution happens at apply-time inside the target stack.

## 1. Setup

In [1]:
%pip install --quiet openpyxl

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import re
import unicodedata
import uuid
from pathlib import Path

import openpyxl

## 2. Configuration

Edit these to point at your xlsx and the deployment dirs for each environment.

In [4]:
XLSX_PATH = Path.home() / "Downloads" / "Overzicht identificatie parameters - Bulk 1 LLB voor ABB.xlsx"

GRAPH_URI = "http://mu.semte.ch/graphs/client-configurations"

# One entry per environment. `client_id_column` is the xlsx column that holds the
# OAuth client id for that environment; `output_dir` is the dev-migrations folder
# in the target deployment. Add/remove environments as needed.
ENVIRONMENTS = {
    "ti": {
        "client_id_column": "CliëntID TEST",
        "output_dir": Path("dev-migrations/ti"),
    },
    "prd": {
        "client_id_column": "CliëntID PRD",
        "output_dir": Path("dev-migrations/prd"),
    },
}

# Filename template. {slug} = slugified Klant, {env} = ti|prd. The brugge example
# in vl-acc uses `<klant>-client-ti.{ttl,graph}`; we keep the basename and emit
# a single .sparql file instead.
FILENAME_TEMPLATE = "{slug}-client-{env}.sparql"

# Any existing file matching this glob (per env, per slug) counts as 'already done' and is skipped.
# Catches both the new .sparql output and any pre-existing .ttl pair (e.g. brugge-client-ti.ttl).
EXISTING_GLOB_TEMPLATE = "{slug}-client-{env}.*"

## 3. Helpers

In [ ]:
def slugify(value: str) -> str:
    """Turn 'Puurs-Sint-Amands' into 'puurs-sint-amands', strip diacritics."""
    value = unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode("ascii")
    value = value.lower().strip()
    value = re.sub(r"[^a-z0-9]+", "-", value)
    return value.strip("-")


def read_xlsx(path: Path) -> list[dict]:
    """Return the first sheet as a list of dicts keyed by header row, skipping blank rows."""
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb[wb.sheetnames[0]]
    rows = ws.iter_rows(values_only=True)
    headers = [str(h).strip() if h is not None else "" for h in next(rows)]
    out = []
    for raw in rows:
        row = {h: (str(v).strip() if v is not None else "") for h, v in zip(headers, raw)}
        if not row.get("KBOnummer") and not row.get("Klant"):
            continue
        out.append(row)
    return out


def render_migration(kbo: str, client_id: str, klant: str, env: str, oauth_config_uri: str) -> str:
    """Return the body of the .sparql migration file.

    At apply-time, the WHERE clause walks adms:identifier → structured
    identifier → lokaleIdentificator to find the bestuurseenheid for the
    given KBO. The oauth-config URI is generated fresh per migration and
    is therefore baked into the INSERT block as a literal IRI.
    """
    return f'''# {klant} ({env}) — KBO {kbo}
PREFIX adms:     <http://www.w3.org/ns/adms#>
PREFIX dct:      <http://purl.org/dc/terms/>
PREFIX ext:      <http://mu.semte.ch/vocabularies/ext/>
PREFIX generiek: <https://data.vlaanderen.be/ns/generiek#>
PREFIX skos:     <http://www.w3.org/2004/02/skos/core#>
PREFIX wotsec:   <https://www.w3.org/2019/wot/security#>

INSERT {{
  GRAPH <{GRAPH_URI}> {{
    ?bestuur ext:hasSecurityScheme <{oauth_config_uri}> .
    <{oauth_config_uri}>
      a wotsec:OAuth2SecurityScheme ;
      dct:identifier "{client_id}" .
  }}
}}
WHERE {{
  ?bestuur adms:identifier ?i .
  ?i  generiek:gestructureerdeIdentificator ?gi ;
      skos:notation "KBO nummer" .
  ?gi generiek:lokaleIdentificator "{kbo}" .
}}
'''


def new_oauth_config_uri() -> str:
    return f"http://data.lblod.info/id/oauth-config/{uuid.uuid4()}"

## 4. Load xlsx

In [6]:
rows = read_xlsx(XLSX_PATH)
print(f"{len(rows)} rows from {XLSX_PATH.name}")
for r in rows[:3]:
    print(r)

48 rows from Overzicht identificatie parameters - Bulk 1 LLB voor ABB.xlsx
{'KBOnummer': '0207502596', 'Klant': 'Kasterlee', 'TAN': '4619p', 'URI TEST': 'kasterlee.be/abb/ipdc37758-aip', 'URI PRD': 'kasterlee.be/abb/ipdc37758', 'CliëntID TEST': 'e43f6fb3-977a-4773-bf18-f61fac0f6f8e', 'CliëntID PRD': '47d6ddb4-7cca-43d2-a5a6-724b65d6cd99'}
{'KBOnummer': '0207537834', 'Klant': 'Stabroek', 'TAN': '4620p', 'URI TEST': 'stabroek.be/abb/ipdc37758-aip', 'URI PRD': 'stabroek.be/abb/ipdc37758', 'CliëntID TEST': '83ad0601-03a8-4976-9670-06c54dbdf453', 'CliëntID PRD': '9c74b845-f226-4cd6-bf85-ec18a80fe492'}
{'KBOnummer': '0207489037', 'Klant': 'Izegem', 'TAN': '4621p', 'URI TEST': 'izegem.be/abb/ipdc37758-aip', 'URI PRD': 'izegem.be/abb/ipdc37758', 'CliëntID TEST': 'dc899f35-c559-464d-865d-76de521bbfaf', 'CliëntID PRD': 'f0427f12-01f0-44f4-9cde-444a2c3657a1'}


## 5. Generate migrations

Per environment, per row: write `<output_dir>/<slug>-client-<env>.sparql` if no file matching `<slug>-client-<env>.*` already exists. Rows missing a KBO, Klant, or the env's client id are reported and skipped.

In [ ]:
DRY_RUN = False  # set True to preview without writing

for env, env_cfg in ENVIRONMENTS.items():
    out_dir: Path = env_cfg["output_dir"]
    client_col: str = env_cfg["client_id_column"]
    print(f"\n=== {env}  →  {out_dir} ===")
    out_dir.mkdir(parents=True, exist_ok=True)

    written, skipped_existing, skipped_invalid = 0, 0, 0
    for r in rows:
        kbo = r["KBOnummer"]
        klant = r["Klant"]
        client_id = r.get(client_col, "")
        if not (kbo and klant and client_id):
            skipped_invalid += 1
            continue

        slug = slugify(klant)
        existing = list(out_dir.glob(EXISTING_GLOB_TEMPLATE.format(slug=slug, env=env)))
        if existing:
            skipped_existing += 1
            continue

        target = out_dir / FILENAME_TEMPLATE.format(slug=slug, env=env)
        body = render_migration(
            kbo=kbo,
            client_id=client_id,
            klant=klant,
            env=env,
            oauth_config_uri=new_oauth_config_uri(),
        )
        if DRY_RUN:
            print(f"  [dry-run] {target.name}")
        else:
            target.write_text(body)
            print(f"  wrote {target.name}")
        written += 1

    print(f"  summary: wrote={written}  skipped_existing={skipped_existing}  skipped_invalid={skipped_invalid}")

## 6. Preview a single migration

Sanity-check the rendered SPARQL for the first row that has a TEST client id.

In [ ]:
for r in rows:
    if r.get("CliëntID TEST"):
        print(render_migration(
            kbo=r["KBOnummer"],
            client_id=r["CliëntID TEST"],
            klant=r["Klant"],
            env="ti",
            oauth_config_uri=new_oauth_config_uri(),
        ))
        break